In [ ]:
#PHENOTYPE TRANSITIONS FROM BASELINE

"""This script charts the movement between groups across the 20 years of follow up, indexed at basleine.""" 


from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection

STATES = ORDER = ['Healthy', 'Borderline', 'Impaired', 'Dementia']


# DESIGN CONFIGURATION

# viridis, sampled light -> dark with increasing severity
STATE_COLORS = {
    'Healthy':    '#bddf26',
    'Borderline': '#26ad81',
    'Impaired':   '#306a8e',
    'Dementia':   '#481769',
}

FIG_SIZE = (25, 12)         # oversized canvas; scale down as SVG afterwards
FS       = 2.2              # font and line scale for that canvas

mpl.rcParams.update({
    'font.family': 'Helvetica', 'font.size': 10 * FS, 'axes.titlesize': 13 * FS,
    'axes.labelsize': 11 * FS, 'xtick.labelsize': 9 * FS, 'ytick.labelsize': 9 * FS,
    'legend.fontsize': 9 * FS, 'svg.fonttype': 'path',
    'pdf.fonttype': 42, 'ps.fonttype': 42,
})

OUTPUT_DIR = Path('./figures')
OUTPUT_FORMAT, OUTPUT_DPI, SAVE_FIGURES = 'svg', 600, True


def save_fig(fig, name, sub=None):
    if not SAVE_FIGURES:
        return
    d = OUTPUT_DIR / sub if sub else OUTPUT_DIR
    d.mkdir(parents=True, exist_ok=True)
    out = d / f'{name}.{OUTPUT_FORMAT}'
    fig.savefig(out, format=OUTPUT_FORMAT, dpi=OUTPUT_DPI, bbox_inches='tight')
    print(f'saved: {out}')

In [ ]:

# PATHS AND PARAMETERS

DATA_PATH = Path('cv_results_all_tests_merged_with_groups.csv')

# --- cohort and state definition (from 05b_group_definition) ---------------
IC_MIN_IMPAIRED = 2        # >= this many impaired tests in one instance -> Impaired
BORDERLINE_MIN   = 1        # >= this many (and < IC_MIN_IMPAIRED)      -> Borderline
MIN_TESTS        = 1        # instances with fewer tests are dropped
MIN_VISITS       = 1        # people with fewer assessments are dropped
DROP_PREVALENT   = False
DROP_EXTREME     = False

# --- time grid -------------------------------------------------------------
YRS_BINS   = list(range(0, 21))     # years since baseline, annual
YRS_WINDOW = 0.5                    # an assessment lands in the column within +/- this
MONOTONE   = False                   # a state can never revert to a milder one
TRUNCATE   = False                  # blank cells past age_at_followup_end

# --- appearance ------------------------------------------------------------
STREAM_GAP  = 0.020         # vertical gap between streams, as a fraction of the cohort
MIN_FLOW    = 0.0           # floor on transition thickness; 0 = area-proportional
EDGE_LW     = 0.15           # ribbon outline — what makes single-person streams visible
ANNOT_YEARS = [0, 5, 10, 15, 20]
ANNOT_COLOR = '#000000'
GUIDE_COLOR = '#555555'

In [ ]:

# LOAD AND REDUCE
"""long (eid x test x instance) -> one state per person per instance"""

long = pd.read_csv(DATA_PATH)
print(f'loaded {DATA_PATH.name}   rows={len(long):,}   eids={long["eid"].nunique():,}')

d = long
if DROP_PREVALENT:
    d = d[d['prevalent'] != 1]
if DROP_EXTREME:
    d = d[d['extreme'] != 1]

visit = (d.groupby(['eid', 'instance'])
           .agg(n_impaired=('impaired', 'sum'), n_tests=('impaired', 'count'),
                age=('age_at_test', 'mean'))
           .reset_index()
           .query('n_tests >= @MIN_TESTS'))

person = d.groupby('eid').agg(age_dx=('age_at_dx', 'first'),
                              age_end=('age_at_followup_end', 'first'),
                              dem=('dem_all', 'first'))

visit['state'] = np.select(
    [visit['n_impaired'] >= IC_MIN_IMPAIRED, visit['n_impaired'] >= BORDERLINE_MIN],
    ['Impaired', 'Borderline'], default='Healthy')
visit = visit.join(person['age_dx'], on='eid')
visit.loc[visit['age'] >= visit['age_dx'], 'state'] = 'Dementia'

print(f'people={len(person):,}   instances={len(visit):,}   '
      f'ever-dementia={int(person["dem"].sum()):,}')
print(visit['state'].value_counts().reindex(STATES).fillna(0).astype(int).to_string())

In [ ]:

# HELPERS

def build_grid(visit, person, ref, window):
    """One state per person per column. Returns S (carried forward), T (assessments)."""
    S = pd.DataFrame(index=ref.index, columns=ref.columns, dtype=object)
    T = pd.DataFrame(index=ref.index, columns=ref.columns, dtype=float)
    for c in ref.columns:
        m = visit.merge(ref[c].rename('tgt'), left_on='eid', right_index=True)
        m = m.assign(d=(m['age'] - m['tgt']).abs()).query('d <= @window')
        m = m.sort_values('d').drop_duplicates('eid')
        S.loc[m['eid'].values, c] = m['state'].values
        T.loc[m['eid'].values, c] = m['age'].values

    S = S.ffill(axis=1)
    if TRUNCATE:
        S = S.where(ref.le(person['age_end'].reindex(ref.index), axis=0))
    S = S.mask(ref.ge(person['age_dx'].reindex(ref.index), axis=0)
               & S.notna().cummax(axis=1), 'Dementia')

    if MONOTONE:
        rank = {s: i for i, s in enumerate(ORDER)}
        code = np.fmax.accumulate(S.replace(rank).astype(float).to_numpy(), axis=1)
        lab = np.where(np.isnan(code), None, np.array(ORDER + [None])[
            np.nan_to_num(code, nan=len(ORDER)).astype(int)])
        S = pd.DataFrame(lab, index=S.index, columns=S.columns)
    return S, T


def _layout(S):
    """Fixed centreline per state. Returns node positions per column and total span."""
    cols = list(S.columns)
    counts = [{s: int((S[c] == s).sum()) for s in ORDER} for c in cols]
    peak = {s: max(d[s] for d in counts) for s in ORDER}
    gap = STREAM_GAP * max(S[c].notna().sum() for c in cols)

    y, centre = 0.0, {}
    for s in ORDER:
        if peak[s]:
            centre[s] = y + peak[s] / 2
            y += peak[s] + gap
    pos = [{s: (centre[s] - d[s] / 2, d[s]) for s in ORDER if d[s]} for d in counts]
    return pos, y


def _band(ax, x0, x1, y0, y1, h, c0, c1, zorder, nseg=48):
    """Ribbon fading from the source colour to the destination colour."""
    mx = (x0 + x1) / 2
    t = np.linspace(0, 1, nseg + 1)
    w = np.column_stack([(1 - t) ** 3, 3 * (1 - t) ** 2 * t, 3 * (1 - t) * t ** 2, t ** 3])
    x = w @ np.array([x0, mx, mx, x1])
    yt = w @ np.array([y0, y0, y1, y1])
    a, b = np.array(mpl.colors.to_rgb(c0)), np.array(mpl.colors.to_rgb(c1))
    quads = [[(x[i], yt[i]), (x[i + 1], yt[i + 1]),
              (x[i + 1], yt[i + 1] + h), (x[i], yt[i] + h)] for i in range(nseg)]
    cols = a + (b - a) * ((t[:-1] + t[1:]) / 2)[:, None]
    ax.add_collection(PolyCollection(quads, facecolors=cols, edgecolors=cols,
                                     linewidths=EDGE_LW * FS, zorder=zorder))


def sankey(S, title, name, xlabel='Years since baseline'):
    cols = list(S.columns)
    pos, span = _layout(S)
    tot = max(S[c].notna().sum() for c in cols)
    xs = np.linspace(0.02, 0.98, len(cols))
    cur = [{s: v[0] for s, v in p.items()} for p in pos]
    inc = [{s: v[0] for s, v in p.items()} for p in pos]

    fig, ax = plt.subplots(figsize=FIG_SIZE)

    for i in range(len(cols) - 1):
        f = pd.DataFrame({'a': S[cols[i]], 'b': S[cols[i + 1]]}).dropna()
        g = f.groupby(['a', 'b']).size().reset_index(name='n')
        g['w'] = np.where(g['a'] != g['b'], np.maximum(g['n'], MIN_FLOW * tot), g['n'])
        g['oa'] = g['a'].map(ORDER.index)
        g['ob'] = g['b'].map(ORDER.index)
        for _, r in g.sort_values(['oa', 'ob']).iterrows():
            y0, y1 = cur[i][r['a']], inc[i + 1][r['b']]
            cur[i][r['a']] += r['w']
            inc[i + 1][r['b']] += r['w']
            # transitions sit above the streams so crossings occlude rather than blend
            _band(ax, xs[i], xs[i + 1], y0, y1, r['w'],
                  STATE_COLORS[r['a']], STATE_COLORS[r['b']],
                  zorder=1 if r['a'] == r['b'] else 3)

    for i, c in enumerate(cols):
        if c not in ANNOT_YEARS:
            continue
        ax.plot([xs[i]] * 2, [-0.02 * span, span], ls=':', lw=1.0 * FS,
                color='#555555', alpha=0.6, zorder=5)
        obs = S[c].dropna()
        txt = '\n'.join(f'{s} {100 * (obs == s).mean():.1f}%'
                        for s in STATES if (obs == s).any())
        ha = 'right' if i == len(cols) - 1 else ('left' if i == 0 else 'center')
        ax.text(xs[i], -0.05 * span, txt, ha=ha, va='bottom', fontsize=9.5 * FS,
                color=ANNOT_COLOR, linespacing=1.45)

    ti = [i for i, c in enumerate(cols) if c in ANNOT_YEARS]
    ax.set_xticks(xs[ti])
    ax.set_xticklabels([str(cols[i]) for i in ti])
    ax.tick_params(axis='x', length=0, pad=2 * FS)
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.set_xlim(0, 1)
    ax.set_ylim(span * 1, -0.20 * span)
    ax.set_xlabel(xlabel)
    ax.set_title(title, pad=19 * FS)
    ax.legend(handles=[mpl.patches.Patch(fc=STATE_COLORS[s], ec='none', label=s)
                       for s in STATES],
              loc='center left', bbox_to_anchor=(0.98, 0.65), frameon=False,
              handlelength=1.1, labelspacing=0.7)
    save_fig(fig, name, sub='transitions')
    plt.show()


BASE_AGE = visit.sort_values('instance').groupby('eid')['age'].first()

In [ ]:
# FIGURE — phenotype transitions from baseline

REF = pd.DataFrame({t: BASE_AGE + t for t in YRS_BINS})
S, T = build_grid(visit, person, REF, YRS_WINDOW)
S = S[T.notna().sum(axis=1) >= MIN_VISITS]

print(f'n={len(S):,}   assessed columns={[t for t in YRS_BINS if T[t].notna().any()]}')
print(S[ANNOT_YEARS].apply(lambda c: c.value_counts()).fillna(0).astype(int).to_string())

sankey(S, 'Cognitive phenotype transitions across time', 'sankey_baseline_years')